# 📄 Descripción general del código

 Este script construye automáticamente un dataset para entrenar un modelo Cross-Encoder
 (por ejemplo, XLM-RoBERTa) para clasificación de relaciones Claim–Evidence.

 A partir de múltiples archivos Excel con afirmaciones y soportes,
 el script genera ejemplos positivos y negativos en formato (sentence1, sentence2, label),

 donde la etiqueta puede ser:
   - 2 → Completamente soportada
   - 1 → Parcialmente soportada
   - 0 → No soportada (parejas negativas)

Finalmente, los datos se guardan como archivo CSV en Google Drive.

In [ ]:
import pandas as pd
import random
from pathlib import Path

In [ ]:
# 📂 Montar Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 📂 Ruta de la carpeta donde están los archivos .xlsx

folder_path = Path('data/annotated_pairs')  # Ajustar según el entorno

In [ ]:
# 📚 Leer todos los archivos .xlsx de la carpeta

file_paths = list(folder_path.glob('*.xlsx'))

In [ ]:
# 🛠️ Recolectar todos los datos válidos

all_data = []

for path in file_paths:
    try:
        df = pd.read_excel(path, sheet_name='claims')
        expected_cols = ['applicant_id', 'answer_id', 'claim_id', 'claim','support', 'relation']
        if all(col in df.columns for col in expected_cols):
            all_data.append(df[expected_cols])
    except Exception as e:
        print(f"Error reading {path}: {e}")

In [ ]:
# ✨ Combinar todos los datos

full_df = pd.concat(all_data, ignore_index=True)
full_df = full_df.dropna(subset=['claim'])

In [ ]:
# ✨ Mapear las relaciones a etiquetas numéricas

relacion_map = {
    'Completamente soportada': 2,
    'Parcialmente soportada': 1,
    'Parcialmente Soportada': 1
}
full_df['label'] = full_df['relation'].map(relacion_map)

In [ ]:
# ✨ Validar afirmaciones (evitar solo símbolos o valores vacíos)

def is_valid_sentence(sentence):
    if not isinstance(sentence, str):
        return False
    if all(c in '.,!?¿¡@#$%^&*()[]{}<>/\\| ' for c in sentence.strip()):
        return False
    return True

In [ ]:
# 🛠️ Agrupar por (applicant_id, answer_id)

grouped = full_df.groupby(['applicant_id', 'answer_id'])

In [ ]:
# 🎯 Lista final de datos para entrenamiento

final_data = []

for (applicant, pregunta), group in grouped:
    for idx, row in group.iterrows():
        afirmacion = row['claim']
        soporte = row['support']
        label = row['label']

        if not is_valid_sentence(afirmacion):
            continue

        if pd.notna(soporte) and label in [1, 2]:  # Crear positivo
            if is_valid_sentence(soporte):
                final_data.append({
                    'sentence1': afirmacion,
                    'sentence2': soporte,
                    'label': label  # Puede ser 1 o 2
                })

            # Crear negativo con otro soporte del grupo

            negative_candidates = group[group['support'] != soporte]['support'].dropna().tolist()
            negative_candidates = [s for s in negative_candidates if is_valid_sentence(s)]

            if negative_candidates:
                negative_soporte = random.choice(negative_candidates)
                final_data.append({
                    'sentence1': afirmacion,
                    'sentence2': negative_soporte,
                    'label': 0
                })

        else:
         # Si no hay soporte → solo crear negativo
            negative_candidates = group['support'].dropna().tolist()
            negative_candidates = [s for s in negative_candidates if is_valid_sentence(s)]

            if negative_candidates:
                negative_soporte = random.choice(negative_candidates)
                final_data.append({
                    'sentence1': afirmacion,
                    'sentence2': negative_soporte,
                    'label': 0
                })

# 📂 Convertir a DataFrame

final_df = pd.DataFrame(final_data)

In [ ]:
# 💾 Guardar en archivo CSV

output_path = 'data/cross_encoder_train_data.csv'
final_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ Cross-encoder training set saved: {output_path}")
print(f"✅ Samples generated: {len(final_df)}")
# print(f"✅ Archivo CSV guardado correctamente: {output_path}")
# print(f"✅ Total de ejemplos generados: {len(final_df)}")

✅ Cross-encoder training set saved: data/cross_encoder_train_data.csv
✅ Samples generated: 6094
